# What's Gained with the Full 73-Book Canon

This notebook analyzes the **additional cross-reference connections** gained when using the full Catholic 73-book canon versus the Protestant 66-book canon.

The 7 deuterocanonical books (Tobit, Judith, Wisdom, Sirach, Baruch, 1-2 Maccabees) provide connections that enrich the understanding of the rest of Scripture.

**Data Source**: Haydock Catholic Bible Commentary cross-references (since TSK doesn't cover deuterocanonical books)

## Setup & Connection

In [1]:
import sys
sys.path.insert(0, '../..')

from src.db.connection import get_connection
from pyvis.network import Network

In [2]:
conn = get_connection()
conn.connect()
conn.verify()
print("Connected to Neo4j!")

Connected to Neo4j!


In [3]:
# The 7 Deuterocanonical books
DEUTEROCANONICAL_BOOKS = ['TOB', 'JDT', 'WIS', 'SIR', 'BAR', '1MA', '2MA']

# Full names for display
DC_BOOK_NAMES = {
    'TOB': 'Tobit',
    'JDT': 'Judith', 
    'WIS': 'Wisdom',
    'SIR': 'Sirach',
    'BAR': 'Baruch',
    '1MA': '1 Maccabees',
    '2MA': '2 Maccabees'
}

# Color scheme (teal for DC books)
BOOK_COLORS = {
    # Pentateuch - Blue
    "GEN": "#3498db", "EXO": "#3498db", "LEV": "#3498db", "NUM": "#3498db", "DEU": "#3498db",
    # Historical - Green
    "JOS": "#27ae60", "JDG": "#27ae60", "RUT": "#27ae60", "1SA": "#27ae60", "2SA": "#27ae60",
    "1KI": "#27ae60", "2KI": "#27ae60", "1CH": "#27ae60", "2CH": "#27ae60", "EZR": "#27ae60",
    "NEH": "#27ae60", "EST": "#27ae60",
    # Deuterocanonical - Teal (highlighted)
    "TOB": "#16a085", "JDT": "#16a085", "1MA": "#16a085", "2MA": "#16a085",
    "WIS": "#16a085", "SIR": "#16a085", "BAR": "#16a085",
    # Wisdom/Poetry - Gold
    "JOB": "#f39c12", "PSA": "#f39c12", "PRO": "#f39c12", "ECC": "#f39c12", "SNG": "#f39c12",
    # Major Prophets - Red
    "ISA": "#e74c3c", "JER": "#e74c3c", "LAM": "#e74c3c", "EZK": "#e74c3c", "DAN": "#e74c3c",
    # Minor Prophets - Orange
    "HOS": "#e67e22", "JOL": "#e67e22", "AMO": "#e67e22", "OBA": "#e67e22", "JON": "#e67e22",
    "MIC": "#e67e22", "NAM": "#e67e22", "HAB": "#e67e22", "ZEP": "#e67e22", "HAG": "#e67e22",
    "ZEC": "#e67e22", "MAL": "#e67e22",
    # Gospels - Purple
    "MAT": "#9b59b6", "MRK": "#9b59b6", "LUK": "#9b59b6", "JHN": "#9b59b6",
    # Acts - Light Purple
    "ACT": "#8e44ad",
    # Pauline Epistles - Pink
    "ROM": "#e91e63", "1CO": "#e91e63", "2CO": "#e91e63", "GAL": "#e91e63", "EPH": "#e91e63",
    "PHP": "#e91e63", "COL": "#e91e63", "1TH": "#e91e63", "2TH": "#e91e63", "1TI": "#e91e63",
    "2TI": "#e91e63", "TIT": "#e91e63", "PHM": "#e91e63",
    # General Epistles - Cyan
    "HEB": "#00bcd4", "JAS": "#00bcd4", "1PE": "#00bcd4", "2PE": "#00bcd4",
    "1JN": "#00bcd4", "2JN": "#00bcd4", "3JN": "#00bcd4", "JUD": "#00bcd4",
    # Revelation - Dark Red
    "REV": "#c0392b",
}

def get_book_color(book_id):
    return BOOK_COLORS.get(book_id, "#95a5a6")

## The Gain Summary

What do Bible readers gain by including the deuterocanonical books?

In [4]:
with conn.session() as session:
    # Total Haydock edges
    result = session.run("""
        MATCH ()-[r:CROSS_REFERENCES]->()
        WHERE 'Haydock' IN r.sources
        RETURN count(r) AS total
    """)
    total_haydock = result.single()['total']
    
    # Haydock edges involving DC books
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND (a.book_id IN dc_books OR b.book_id IN dc_books)
        RETURN count(r) AS gained
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    gained_edges = result.single()['gained']
    
    # Edges internal to DC books
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND a.book_id IN dc_books AND b.book_id IN dc_books
        RETURN count(r) AS internal
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    internal_edges = result.single()['internal']
    
    # Bridging edges (DC to non-DC)
    bridging_edges = gained_edges - internal_edges
    
    # Verses in shared canon that gain connections
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]-(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND NOT a.book_id IN dc_books AND b.book_id IN dc_books
        RETURN count(DISTINCT a) AS verses_enriched
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    verses_enriched = result.single()['verses_enriched']
    
    print("="*60)
    print("WHAT YOU GAIN WITH THE FULL 73-BOOK CANON")
    print("="*60)
    print(f"\nTotal Haydock cross-references:       {total_haydock:>10,}")
    print(f"\nGained from deuterocanonical books:   {gained_edges:>10,} ({gained_edges/total_haydock*100:.1f}%)")
    print(f"  - Within DC books:                  {internal_edges:>10,}")
    print(f"  - Connecting DC to shared canon:    {bridging_edges:>10,}")
    print(f"\nVerses in shared canon enriched:      {verses_enriched:>10,}")
    print("\n" + "="*60)

WHAT YOU GAIN WITH THE FULL 73-BOOK CANON

Total Haydock cross-references:            6,582

Gained from deuterocanonical books:          857 (13.0%)
  - Within DC books:                         103
  - Connecting DC to shared canon:           754

Verses in shared canon enriched:             501



## Gained Connections by Deuterocanonical Book

How much does each deuterocanonical book contribute?

In [5]:
with conn.session() as session:
    # Get connections per DC book
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]-(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND a.book_id IN dc_books
        RETURN a.book_id AS dc_book, count(DISTINCT r) AS connections
        ORDER BY connections DESC
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    
    print("Cross-References Contributed by Each Deuterocanonical Book:")
    print("="*55)
    print(f"{'Book':6} {'Name':15} {'Connections':>15}")
    print("-"*55)
    
    total = 0
    for record in result:
        book = record['dc_book']
        name = DC_BOOK_NAMES.get(book, book)
        conns = record['connections']
        total += conns
        print(f"{book:6} {name:15} {conns:>15,}")
    
    print("-"*55)
    print(f"{'':6} {'TOTAL':15} {total:>15,}")

Cross-References Contributed by Each Deuterocanonical Book:
Book   Name                Connections
-------------------------------------------------------
SIR    Sirach                      446
WIS    Wisdom                      214
TOB    Tobit                        70
1MA    1 Maccabees                  61
2MA    2 Maccabees                  48
BAR    Baruch                       30
JDT    Judith                       29
-------------------------------------------------------
       TOTAL                       898


## Books Most Enriched

Which books in the shared canon gain the most connections from the deuterocanonical books?

In [6]:
with conn.session() as session:
    result = session.run("""
        WITH $dc_books AS dc_books
        MATCH (a:Verse)-[r:CROSS_REFERENCES]-(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND NOT a.book_id IN dc_books 
          AND b.book_id IN dc_books
        RETURN a.book_id AS book, count(r) AS gained_connections
        ORDER BY gained_connections DESC
        LIMIT 20
    """, dc_books=DEUTEROCANONICAL_BOOKS)
    
    print("Books Most Enriched by Deuterocanonical Connections:")
    print("="*45)
    print(f"{'Rank':5} {'Book':8} {'Gained Connections':>20}")
    print("-"*45)
    for i, record in enumerate(result, 1):
        print(f"{i:5} {record['book']:8} {record['gained_connections']:>20,}")

Books Most Enriched by Deuterocanonical Connections:
Rank  Book       Gained Connections
---------------------------------------------
    1 GEN                        88
    2 DEU                        67
    3 EXO                        64
    4 PRO                        52
    5 ISA                        38
    6 PSA                        38
    7 NUM                        37
    8 2KI                        30
    9 1SA                        26
   10 LEV                        25
   11 ROM                        23
   12 MAT                        23
   13 1KI                        23
   14 JER                        16
   15 JHN                        15
   16 DAN                        14
   17 2CH                        14
   18 JOB                        14
   19 LUK                        13
   20 HEB                        13


## Gained Thematic Connections

Let's look at actual examples of what connections are gained.

### Wisdom Literature Connections

How Sirach and Wisdom connect to Proverbs, Psalms, and other wisdom books.

In [17]:
with conn.session() as session:
    # Wisdom connections to wisdom literature
    wisdom_books = ['PRO', 'PSA', 'JOB', 'ECC', 'SNG']
    dc_wisdom = ['WIS', 'SIR']
    
    result = session.run("""
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND a.book_id IN $dc_wisdom
          AND b.book_id IN $wisdom_books
        RETURN a.id AS from_id, a.book_name AS from_book, a.text AS from_text,
               b.id AS to_id, b.book_name AS to_book, b.text AS to_text
        LIMIT 8
    """, dc_wisdom=dc_wisdom, wisdom_books=wisdom_books)
    
    print("Wisdom Literature Connections (Sirach/Wisdom -> Proverbs/Psalms/etc):")
    print("="*80)
    for record in result:
        print(f"\n{record['from_id']} ({record['from_book']})")
        print(f"  -> {record['to_id']} ({record['to_book']})")
        print(f"  FROM: {record['from_text'][:200]}...")
        print(f"  TO:   {record['to_text'][:200]}...")

Wisdom Literature Connections (Sirach/Wisdom -> Proverbs/Psalms/etc):

WIS-2-1 (Wisdom)
  -> JOB-14-1 (Job)
  FROM: For they have said, reasoning with themselves incorrectly: “Our lifetime is brief and tedious, and there is no relief within the limits of man, and no one is acknowledged to have returned from the dea...
  TO:   Man, born of woman, living for a short time, is filled with many miseries....

WIS-2-1 (Wisdom)
  -> JOB-7-1 (Job)
  FROM: For they have said, reasoning with themselves incorrectly: “Our lifetime is brief and tedious, and there is no relief within the limits of man, and no one is acknowledged to have returned from the dea...
  TO:   The life of a man on the earth is a battle, and his days are like the days of a hired hand....

WIS-2-14 (Wisdom)
  -> PSA-21-9 (Psalms)
  FROM: He was made among us to expose our very thoughts....
  TO:   He has hoped in the Lord, let him rescue him. Let him save him because he chooses him....

WIS-5-10 (Wisdom)
  -> PRO-30-19 (Prover

### Historical Connections

How Maccabees and other historical DC books connect to Kings, Chronicles, Daniel.

In [18]:
with conn.session() as session:
    historical_books = ['1KI', '2KI', '1CH', '2CH', 'DAN', 'EZR', 'NEH']
    dc_historical = ['1MA', '2MA', 'TOB', 'JDT']
    
    result = session.run("""
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND a.book_id IN $dc_historical
          AND b.book_id IN $historical_books
        RETURN a.id AS from_id, a.book_name AS from_book, a.text AS from_text,
               b.id AS to_id, b.book_name AS to_book, b.text AS to_text
        LIMIT 8
    """, dc_historical=dc_historical, historical_books=historical_books)
    
    print("Historical Connections (Maccabees/Tobit/Judith -> Kings/Chronicles/Daniel):")
    print("="*80)
    for record in result:
        print(f"\n{record['from_id']} ({record['from_book']})")
        print(f"  -> {record['to_id']} ({record['to_book']})")
        print(f"  FROM: {record['from_text'][:200]}...")
        print(f"  TO:   {record['to_text'][:200]}...")

Historical Connections (Maccabees/Tobit/Judith -> Kings/Chronicles/Daniel):

1MA-2-58 (I Maccabees)
  -> 2KI-2-11 (II Kings)
  FROM: Elijah, since he was zealous with a zeal for the law, was received into heaven....
  TO:   And as they continued on, they were conversing while walking. And behold, a fiery chariot with fiery horses divided the two. And Elijah ascended by a whirlwind into heaven....

TOB-1-2 (Tobit)
  -> 2KI-17-3 (II Kings)
  FROM: Although he had been taken captive in the days of Shalmaneser, the king of the Assyrians, even in such a situation as captivity, he did not desert the way of truth....
  TO:   Shalmaneser, the king of the Assyrians, ascended against him. And Hoshea became a servant to him, and he paid him tribute....

TOB-1-2 (Tobit)
  -> 2KI-18-9 (II Kings)
  FROM: Although he had been taken captive in the days of Shalmaneser, the king of the Assyrians, even in such a situation as captivity, he did not desert the way of truth....
  TO:   In the fourth year of 

### Prophetic Connections

How Baruch connects to Jeremiah and other prophets.

In [19]:
with conn.session() as session:
    prophetic_books = ['ISA', 'JER', 'EZK', 'DAN', 'HOS', 'JOL', 'AMO']
    
    result = session.run("""
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND a.book_id = 'BAR'
          AND b.book_id IN $prophetic_books
        RETURN a.id AS from_id, a.book_name AS from_book, a.text AS from_text,
               b.id AS to_id, b.book_name AS to_book, b.text AS to_text
        LIMIT 8
    """, prophetic_books=prophetic_books)
    
    print("Prophetic Connections (Baruch -> Isaiah/Jeremiah/Ezekiel):")
    print("="*80)
    for record in result:
        print(f"\n{record['from_id']} ({record['from_book']})")
        print(f"  -> {record['to_id']} ({record['to_book']})")
        print(f"  FROM: {record['from_text'][:200]}...")
        print(f"  TO:   {record['to_text'][:200]}...")

Prophetic Connections (Baruch -> Isaiah/Jeremiah/Ezekiel):

BAR-1-17 (Baruch)
  -> DAN-9-5 (Daniel)
  FROM: We have sinned before the Lord our God and we have not believed, lacking confidence in him....
  TO:   We have sinned, we have committed iniquity, we acted impiously and have withdrawn, and we have turned aside from your commandments as well as your judgments....

BAR-2-11 (Baruch)
  -> DAN-9-15 (Daniel)
  FROM: And now, O Lord God of Israel, who has led your people out of the land of Egypt with a strong hand, and with signs, and with wonders, and with your great power, and with an exalted arm, and has made a...
  TO:   And now, O Lord, our God, who has led your people out of the land of Egypt with a strong hand and has made yourself a name in accordance with this day: we have sinned, we have done wrong....

BAR-2-16 (Baruch)
  -> ISA-63-15 (Isaiah)
  FROM: Gaze upon us, O Lord, from your holy home, and incline your ear, and heed us....
  TO:   Gaze down from heaven, and behold f

### New Testament Connections

How the deuterocanonical books illuminate the New Testament.

In [20]:
with conn.session() as session:
    nt_books = ['MAT', 'MRK', 'LUK', 'JHN', 'ACT', 'ROM', '1CO', '2CO', 'GAL', 'EPH',
                'PHP', 'COL', '1TH', '2TH', '1TI', '2TI', 'TIT', 'PHM', 'HEB',
                'JAS', '1PE', '2PE', '1JN', '2JN', '3JN', 'JUD', 'REV']
    
    result = session.run("""
        MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
        WHERE 'Haydock' IN r.sources
          AND a.book_id IN $dc_books
          AND b.book_id IN $nt_books
        RETURN a.id AS from_id, a.book_name AS from_book, a.text AS from_text,
               b.id AS to_id, b.book_name AS to_book, b.text AS to_text
        LIMIT 10
    """, dc_books=DEUTEROCANONICAL_BOOKS, nt_books=nt_books)
    
    print("New Testament Connections (DC books -> Gospels/Epistles):")
    print("="*80)
    for record in result:
        print(f"\n{record['from_id']} ({record['from_book']})")
        print(f"  -> {record['to_id']} ({record['to_book']})")
        print(f"  FROM: {record['from_text'][:128]}...")
        print(f"  TO:   {record['to_text'][:128]}...")

New Testament Connections (DC books -> Gospels/Epistles):

SIR-2-1 (Sirach)
  -> 2TI-3-12 (II Timothy)
  FROM: Son, when you apply yourself to the service of God, stand in justice and in fear, and prepare your soul for temptation....
  TO:   And all those who willingly live the piety in Christ Jesus will suffer persecution....

SIR-2-1 (Sirach)
  -> MAT-4-1 (Matthew)
  FROM: Son, when you apply yourself to the service of God, stand in justice and in fear, and prepare your soul for temptation....
  TO:   Then Jesus was led by the Spirit into the desert, in order to be tempted by the devil....

SIR-2-18 (Sirach)
  -> JHN-14-23 (John)
  FROM: Those who fear the Lord will not be unbelieving toward his Word. And those who love him will keep to his way....
  TO:   Jesus responded and said to him: “If anyone loves me, he shall keep my word. And my Father will love him, and we will come to hi...

SIR-3-9 (Sirach)
  -> MRK-7-10 (Mark)
  FROM: In word and deed, and in all things, honor your fath

## Visualization: The Enriched Network

Visualize how the deuterocanonical books connect to and enrich the rest of Scripture.

In [11]:
def visualize_dc_enrichment(min_connections=10):
    """Create a book-level graph showing how DC books enrich the canon.
    
    DC books are highlighted in teal and larger.
    """
    net = Network(
        height="700px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white",
        notebook=True,
        cdn_resources='in_line'
    )
    net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=300)
    
    # Query book-level connections involving DC books
    query = """
    WITH $dc_books AS dc_books
    MATCH (a:Verse)-[r:CROSS_REFERENCES]->(b:Verse)
    WHERE 'Haydock' IN r.sources
      AND (a.book_id IN dc_books OR b.book_id IN dc_books)
      AND a.book_id <> b.book_id
    WITH a.book_id AS from_book, b.book_id AS to_book, count(r) AS connections
    WHERE connections >= $min_connections
    RETURN from_book, to_book, connections
    ORDER BY connections DESC
    """
    
    added_nodes = set()
    
    with conn.session() as session:
        results = session.run(query, dc_books=DEUTEROCANONICAL_BOOKS, min_connections=min_connections)
        
        for record in results:
            from_book = record["from_book"]
            to_book = record["to_book"]
            connections = record["connections"]
            
            # Add book nodes
            for book in [from_book, to_book]:
                if book not in added_nodes:
                    is_dc = book in DEUTEROCANONICAL_BOOKS
                    label = DC_BOOK_NAMES.get(book, book) if is_dc else book
                    net.add_node(
                        book,
                        label=label,
                        title=f"{label} (Deuterocanonical)" if is_dc else label,
                        color=get_book_color(book),
                        size=40 if is_dc else 25,
                        borderWidth=4 if is_dc else 1,
                        font={'size': 16 if is_dc else 12},
                    )
                    added_nodes.add(book)
            
            # Add edge (teal for all since they involve DC)
            net.add_edge(
                from_book,
                to_book,
                title=f"{connections:,} connections",
                width=max(1, connections / 10),
                color="#16a085",
            )
    
    print(f"Graph: {len(added_nodes)} books")
    print("Larger teal nodes = Deuterocanonical books")
    return net

In [12]:
net = visualize_dc_enrichment(min_connections=5)
net.show("dc_enrichment_network.html")

Graph: 27 books
Larger teal nodes = Deuterocanonical books
dc_enrichment_network.html


### Verse-Level Network: Sirach Connections

Sirach is often the most connected deuterocanonical book. Let's visualize connections around a key verse.

In [13]:
def visualize_verse_connections(verse_id, limit=30):
    """Visualize Haydock connections around a specific verse."""
    net = Network(
        height="600px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white",
        notebook=True,
        cdn_resources='in_line'
    )
    net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=200)
    
    query = """
    MATCH (center:Verse {id: $verse_id})
    OPTIONAL MATCH (center)-[r:CROSS_REFERENCES]-(connected:Verse)
    WHERE 'Haydock' IN r.sources
    WITH center, connected, r
    LIMIT $limit
    RETURN center.id AS center_id, center.book_id AS center_book, 
           center.text AS center_text,
           connected.id AS conn_id, connected.book_id AS conn_book,
           connected.text AS conn_text
    """
    
    added_nodes = set()
    
    with conn.session() as session:
        results = session.run(query, verse_id=verse_id, limit=limit)
        
        for record in results:
            center_id = record["center_id"]
            if center_id not in added_nodes:
                is_dc = record["center_book"] in DEUTEROCANONICAL_BOOKS
                net.add_node(
                    center_id,
                    label=center_id,
                    title=record["center_text"][:200],
                    color=get_book_color(record["center_book"]),
                    size=35,
                    borderWidth=4 if is_dc else 1,
                )
                added_nodes.add(center_id)
            
            conn_id = record["conn_id"]
            if conn_id and conn_id not in added_nodes:
                is_dc = record["conn_book"] in DEUTEROCANONICAL_BOOKS
                net.add_node(
                    conn_id,
                    label=conn_id,
                    title=record["conn_text"][:200] if record["conn_text"] else "",
                    color=get_book_color(record["conn_book"]),
                    size=20,
                    borderWidth=3 if is_dc else 1,
                )
                added_nodes.add(conn_id)
            
            if conn_id:
                net.add_edge(center_id, conn_id, color="#16a085", width=2)
    
    print(f"Graph: {len(added_nodes)} nodes around {verse_id}")
    return net

In [14]:
# Sirach 6:14 - "A faithful friend is a sturdy shelter"
net = visualize_verse_connections("SIR-6-14", limit=25)
net.show("sirach_6_14_connections.html")

Graph: 1 nodes around SIR-6-14
sirach_6_14_connections.html


In [15]:
# Wisdom 2:24 - "By the envy of the devil, death entered the world"
net = visualize_verse_connections("WIS-2-24", limit=25)
net.show("wisdom_2_24_connections.html")

Graph: 2 nodes around WIS-2-24
wisdom_2_24_connections.html


## Summary

The 7 deuterocanonical books provide:

1. **Additional cross-references** that connect to and illuminate the rest of Scripture
2. **Wisdom literature enrichment** - Sirach and Wisdom expand on Proverbs and Psalms themes
3. **Historical context** - Maccabees fills the intertestamental period, connecting to Daniel and prophetic fulfillment
4. **Prophetic connections** - Baruch continues and clarifies Jeremiah's message
5. **New Testament background** - Many DC passages illuminate Gospel and Epistle themes

These connections represent what readers of the full 73-book canon gain - a richer, more interconnected understanding of Scripture.

## Cleanup

In [16]:
conn.close()
print("Connection closed")

Connection closed
